In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:
BASE_DIR = r"C:\Users\nshre\OneDrive\Desktop\Vision-Based-Fake-Medicine-Detection\dataset"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [4]:
train_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "Train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "Val"),
    transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
print("Classes:", class_names)


Classes: ['Fake', 'Real']


In [5]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

# Load pretrained weights (recommended)
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Replace final fully connected layer
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [7]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader, 1):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Optional: print progress every 5 batches
        if batch_idx % 5 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Batch [{batch_idx}/{len(train_loader)}] "
                  f"Loss: {running_loss/batch_idx:.4f}")

    train_acc = 100 * correct / total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            val_loss += criterion(outputs, labels).item()

    val_acc = 100 * val_correct / val_total
    val_loss_avg = val_loss / len(val_loader)

    print(f"\n✅ Epoch [{epoch+1}/{EPOCHS}] Completed: "
          f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {val_loss_avg:.4f}, Val Acc: {val_acc:.2f}%\n")


Epoch [1/5] Batch [5/42] Loss: 0.5961
Epoch [1/5] Batch [10/42] Loss: 0.5941
Epoch [1/5] Batch [15/42] Loss: 0.5785
Epoch [1/5] Batch [20/42] Loss: 0.5499
Epoch [1/5] Batch [25/42] Loss: 0.5263
Epoch [1/5] Batch [30/42] Loss: 0.5132
Epoch [1/5] Batch [35/42] Loss: 0.4896
Epoch [1/5] Batch [40/42] Loss: 0.4873

✅ Epoch [1/5] Completed: Train Loss: 0.4829, Train Acc: 77.42%, Val Loss: 0.3254, Val Acc: 87.86%

Epoch [2/5] Batch [5/42] Loss: 0.3656
Epoch [2/5] Batch [10/42] Loss: 0.3782
Epoch [2/5] Batch [15/42] Loss: 0.3639
Epoch [2/5] Batch [20/42] Loss: 0.3729
Epoch [2/5] Batch [25/42] Loss: 0.3558
Epoch [2/5] Batch [30/42] Loss: 0.3452
Epoch [2/5] Batch [35/42] Loss: 0.3440
Epoch [2/5] Batch [40/42] Loss: 0.3429

✅ Epoch [2/5] Completed: Train Loss: 0.3456, Train Acc: 87.12%, Val Loss: 0.2343, Val Acc: 94.26%

Epoch [3/5] Batch [5/42] Loss: 0.2753
Epoch [3/5] Batch [10/42] Loss: 0.2933
Epoch [3/5] Batch [15/42] Loss: 0.3191
Epoch [3/5] Batch [20/42] Loss: 0.2930
Epoch [3/5] Batch [25/4

In [8]:
MODEL_PATH = "fake_medicine_model.pth"
torch.save(model.state_dict(), MODEL_PATH)
print("✅ Model saved successfully!")


✅ Model saved successfully!
